# 08 when / otherwise

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Write conditional logic in PySpark with <code>F.when(...).otherwise(...)</code> and SQL <code>CASE WHEN</code>, handle multiple and nested conditions, and avoid the classic traps with order and nulls.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Business rules are conditional: customer segments, risk bands, status mapping, data fixes. <code>when/otherwise</code> is the PySpark version of <code>if/elif/else</code> and of SQL <code>CASE WHEN</code>. You will write it in almost every pipeline, and you should never need a Python UDF for this kind of logic.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 How when / otherwise works</b><br>
<code>F.when(condition, value)</code> starts a conditional expression. Chain more <code>.when(...)</code> calls for extra branches, and finish with <code>.otherwise(default)</code>.
<ul><li>Conditions are checked <b>top to bottom</b>. The <b>first</b> true condition wins</li><li>Without <code>.otherwise()</code>, rows that match nothing get <b>null</b></li><li>A null condition counts as "not true", so it falls through to the next branch</li><li>It runs as a native Spark expression (fast), unlike a Python UDF</li></ul>
</div>

```
F.when(c1, v1)          if c1:      v1
 .when(c2, v2)          elif c2:    v2
 .otherwise(v3)         else:       v3
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A telecom company bands customers every night: <code>platinum</code> if monthly spend ≥ 200, <code>gold</code> if ≥ 100, <code>silver</code> if ≥ 30, otherwise <code>bronze</code>, and <code>unknown</code> if spend is missing. If the conditions are written in the wrong order, every platinum customer ends up as silver. That is a real bug that has shipped to production more than once.
</div>

## Syntax

```python
from pyspark.sql import functions as F

df.withColumn(
    "band",
    F.when(F.col("spend").isNull(), "unknown")
     .when(F.col("spend") >= 200, "platinum")
     .when(F.col("spend") >= 100, "gold")
     .otherwise("bronze"),
)

# SQL equivalent
df.selectExpr("*", "CASE WHEN spend >= 200 THEN 'platinum' ELSE 'other' END AS band")
```

## Setup: sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a customer DataFrame with spend, tenure and status codes, including nulls.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A 7-row table. <code>C6</code> has a null spend and <code>C7</code> has a null status code.
</div>

In [0]:
from pyspark.sql import functions as F

customers = spark.createDataFrame(
    [
        ("C1", "Amal", 250.0, 36, "A", "IN"),
        ("C2", "Senthil", 120.0, 5, "S", "US"),
        ("C3", "John", 45.0, 18, "A", "UK"),
        ("C4", "Vivek", 12.0, 2, "C", "AE"),
        ("C5", "Priya", 199.99, 60, "A", "IN"),
        ("C6", "Sara", None, 7, "S", "AE"),
        ("C7", "Ravi", 80.0, 24, None, "IN"),
    ],
    "customer_id STRING, name STRING, monthly_spend DOUBLE, tenure_months INT, status_code STRING, country STRING",
)
display(customers)

## 1. A single condition

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Flags customers as <code>high</code> or <code>normal</code> with one <code>when</code> and one <code>otherwise</code>.<br><br>
<b>Why it matters</b><br>The simplest form. Notice the null-spend customer: the condition <code>null &gt;= 150</code> is null, so it falls to <code>otherwise</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>C1</code> and <code>C5</code> are <code>high</code>, everyone else (including <code>C6</code> with null spend) is <code>normal</code>.
</div>

In [0]:
customers.withColumn(
    "spend_flag",
    F.when(F.col("monthly_spend") >= 150, "high").otherwise("normal"),
).select("customer_id", "monthly_spend", "spend_flag").show()

## 2. Multiple conditions: order matters

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds a 5-level band, first in the correct order and then in the wrong order.<br><br>
<b>Why it matters</b><br>Only the <b>first</b> matching branch is used. Put the most specific (or highest) conditions first, and handle nulls first if they need their own label.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Correct order: <code>C1</code> platinum, <code>C2</code> gold, <code>C3</code> silver, <code>C4</code> bronze, <code>C5</code> gold, <code>C6</code> unknown, <code>C7</code> silver. Wrong order: everyone with spend ≥ 30 becomes <code>silver</code>.
</div>

In [0]:
correct_band = (
    F.when(F.col("monthly_spend").isNull(), "unknown")
     .when(F.col("monthly_spend") >= 200, "platinum")
     .when(F.col("monthly_spend") >= 100, "gold")
     .when(F.col("monthly_spend") >= 30, "silver")
     .otherwise("bronze")
)

wrong_band = (
    F.when(F.col("monthly_spend") >= 30, "silver")      # too broad, catches everything above it
     .when(F.col("monthly_spend") >= 100, "gold")        # never reached
     .when(F.col("monthly_spend") >= 200, "platinum")    # never reached
     .otherwise("bronze")
)

customers.select(
    "customer_id", "monthly_spend",
    correct_band.alias("correct_band"),
    wrong_band.alias("wrong_band"),
).show()

## 3. Combined and nested conditions

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses <code>&amp;</code> and <code>|</code> inside conditions, then nests a <code>when</code> inside another <code>when</code>.<br><br>
<b>Why it matters</b><br>Real rules combine several columns. Nesting works, but a flat chain with combined conditions is usually easier to read and test.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>offer</code> is <code>loyalty_upgrade</code> for long-tenure high spenders (C1, C5), <code>win_back</code> for low spenders in IN or AE (C4), <code>none</code> otherwise. The nested version gives the same answers as the flat version.
</div>

In [0]:
flat = (
    F.when((F.col("tenure_months") >= 24) & (F.col("monthly_spend") >= 150), "loyalty_upgrade")
     .when((F.col("monthly_spend") < 30) & F.col("country").isin("IN", "AE"), "win_back")
     .otherwise("none")
)

nested = (
    F.when(
        F.col("tenure_months") >= 24,
        F.when(F.col("monthly_spend") >= 150, "loyalty_upgrade").otherwise("none"),
    )
    .when((F.col("monthly_spend") < 30) & F.col("country").isin("IN", "AE"), "win_back")
    .otherwise("none")
)

customers.select("customer_id", "tenure_months", "monthly_spend", "country",
                 flat.alias("offer_flat"), nested.alias("offer_nested")).show()

## 4. Mapping codes to labels, and what happens without otherwise

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Maps status codes <code>A</code>, <code>S</code>, <code>C</code> to readable labels, with and without a default.<br><br>
<b>Why it matters</b><br>Code-to-label mapping is everywhere. Forgetting <code>.otherwise()</code> quietly produces nulls for unexpected codes, which a data quality check should catch.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>status_no_default</code> is null for <code>C7</code> (null code). <code>status</code> shows <code>Unknown</code> for it instead.
</div>

In [0]:
status_label = (
    F.when(F.col("status_code") == "A", "Active")
     .when(F.col("status_code") == "S", "Suspended")
     .when(F.col("status_code") == "C", "Closed")
)

customers.select(
    "customer_id", "status_code",
    status_label.alias("status_no_default"),
    status_label.otherwise("Unknown").alias("status"),
).show()

## 5. The same logic in SQL: CASE WHEN

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the spend band with <code>CASE WHEN</code> through <code>selectExpr</code>, and through a temporary view with <code>spark.sql</code>.<br><br>
<b>Why it matters</b><br>SQL-first teams and Databricks SQL dashboards use <code>CASE WHEN</code>. The exam shows both PySpark and SQL syntax.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Two tables with the same bands as the correct order in section 2.
</div>

In [0]:
customers.selectExpr(
    "customer_id",
    """CASE
         WHEN monthly_spend IS NULL THEN 'unknown'
         WHEN monthly_spend >= 200 THEN 'platinum'
         WHEN monthly_spend >= 100 THEN 'gold'
         WHEN monthly_spend >= 30  THEN 'silver'
         ELSE 'bronze'
       END AS band""",
).show()

customers.createOrReplaceTempView("customers_v")
spark.sql("""
    SELECT customer_id,
           CASE status_code WHEN 'A' THEN 'Active' WHEN 'S' THEN 'Suspended' ELSE 'Other' END AS status
    FROM customers_v
""").show()

## 6. Conditional aggregation

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses <code>when</code> inside <code>sum</code> and <code>count</code> to count customers per category in a single pass.<br><br>
<b>Why it matters</b><br>A very common reporting pattern ("how many active and how many high spenders per country?"). It needs only one aggregation instead of several filtered queries.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
One row per country with <code>customers</code>, <code>active</code> and <code>high_spenders</code> counts. India: 3 customers, 2 active, 3 high spenders (spend of 75 or more).
</div>

In [0]:
customers.groupBy("country").agg(
    F.count("*").alias("customers"),
    F.sum(F.when(F.col("status_code") == "A", 1).otherwise(0)).alias("active"),
    F.count(F.when(F.col("monthly_spend") >= 75, True)).alias("high_spenders"),  # count ignores nulls
).orderBy("country").show()

## 7. Rules from configuration

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds a <code>when</code> chain from a Python list of rules using <code>functools.reduce</code>.<br><br>
<b>Why it matters</b><br>In real projects business rules often live in a config file or table, so the code doesn't change when a threshold does.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The same bands as section 2, produced from the <code>rules</code> list.
</div>

In [0]:
from functools import reduce

rules = [  # (minimum spend, label), checked in this order
    (200, "platinum"),
    (100, "gold"),
    (30, "silver"),
]

expr = F.when(F.col("monthly_spend").isNull(), "unknown")
expr = reduce(lambda acc, rule: acc.when(F.col("monthly_spend") >= rule[0], rule[1]), rules, expr)
expr = expr.otherwise("bronze")

customers.select("customer_id", "monthly_spend", expr.alias("band")).show()

## Under the hood

```
F.when(c1, v1).when(c2, v2).otherwise(v3)
        |
        v
 Catalyst expression:  CaseWhen([(c1, v1), (c2, v2)], else = v3)
        |
        v
 Whole-stage code generation turns it into an if/else in generated Java code
        |
        v
 Evaluated row by row inside each partition (narrow, no shuffle)
```

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> `when` builds an expression. It runs only when an action is called.

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> evaluated per row, no data movement.

Because `CaseWhen` is a built-in expression, Spark can optimize and code-generate it. A Python UDF with `if/else` would ship every row to a Python process and back, which is much slower (lesson 20). Look for `CASE WHEN` in the plan below.

In [0]:
spark.range(100).withColumn(
    "size", F.when(F.col("id") > 50, "big").otherwise("small")
).explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: unexpected nulls in the new column</b><br>
No <code>.otherwise()</code>, so unmatched rows become null. Add a default.<br><br>
<b>⛔ Problem: most rows land in one band</b><br>Conditions in the wrong order. Most specific or highest threshold first.<br><br>
**⛔ Problem: <code>when() can only be applied on a Column previously generated by when()</code>**<br>You called <code>.when()</code> on a normal column, for example <code>F.col("x").when(...)</code>. Start the chain with <code>F.when(...)</code>.<br><br>
<b>⛔ Problem: rows with null inputs get the default instead of their own label</b><br>A null condition is not true. Add an explicit <code>isNull()</code> branch first.<br><br>
<b>⛔ Problem: mixed result types</b><br>Branches returning a string in one place and a number in another are coerced to a common type, or fail under ANSI mode. Keep all branches the same type.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How do you write if/else logic in PySpark?</b><br>
With <code>F.when(condition, value).when(...).otherwise(default)</code>, or with SQL <code>CASE WHEN</code> in <code>selectExpr</code> or <code>spark.sql</code>. Both become the same <code>CaseWhen</code> expression.<br><br>

<b>🎤 2. What happens if no condition matches and there is no <code>otherwise</code>?</b><br>
The result is null for that row.<br><br>

<b>🎤 3. Does the order of <code>when</code> clauses matter?</b><br>
Yes. They are evaluated top to bottom and the first true condition wins, so broader conditions must come after specific ones.<br><br>

<b>🎤 4. How are nulls handled in <code>when</code> conditions?</b><br>
A condition that evaluates to null is treated as not true, so the row moves on to the next branch or to <code>otherwise</code>. Handle nulls explicitly with <code>isNull()</code> if they need their own result.<br><br>

<b>🎤 5. Why prefer <code>when</code> over a Python UDF for conditional logic?</b><br>
<code>when</code> is a native Catalyst expression, so it is optimized and code-generated and runs in the JVM. A UDF serializes each row to Python and back, and the optimizer can't see inside it.<br><br>

<b>🎤 6. How do you count rows matching a condition inside an aggregation?</b><br>
<code>F.sum(F.when(cond, 1).otherwise(0))</code>, or <code>F.count(F.when(cond, True))</code>, because <code>count</code> ignores nulls.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. What is the result for a row with <code>score = 85</code> in <code>CASE WHEN score &gt; 50 THEN 'pass' WHEN score &gt; 80 THEN 'merit' ELSE 'fail' END</code>?</b><br>
A. <code>merit</code><br>
B. <code>pass</code><br>
C. <code>fail</code><br>
D. NULL<br>
<details><summary><b>Show answer</b></summary><b>B.</b> The first true condition wins. <code>score &gt; 50</code> is true, so <code>merit</code> is never evaluated.</details><br><br>

<b>Q2. What does <code>F.when(col("x") &gt; 10, "big")</code> return for rows where <code>x = 3</code>?</b><br>
A. <code>"small"</code><br>
B. An empty string<br>
C. NULL<br>
D. An error<br>
<details><summary><b>Show answer</b></summary><b>C.</b> Without <code>otherwise</code>, unmatched rows are NULL.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b><br>
Using <code>customers</code>:
<ol><li>Add <code>tenure_group</code>: <code>new</code> (< 12 months), <code>established</code> (12 to 35), <code>loyal</code> (36+)</li><li>Add <code>region</code>: <code>APAC</code> for IN, <code>MEA</code> for AE, <code>EMEA</code> for UK, <code>AMER</code> for US, otherwise <code>OTHER</code></li><li>Add <code>priority</code>: <code>P1</code> if status is Active <b>and</b> spend ≥ 100, <code>P2</code> if status is Active, <code>P3</code> for everything else, including null status</li><li>Write task 1 again using SQL <code>CASE WHEN</code> in <code>selectExpr</code></li><li>Count customers per <code>tenure_group</code> and how many of them are from India, in one aggregation</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Tenure groups
tenure_group = (
    F.when(F.col("tenure_months") < 12, "new")
     .when(F.col("tenure_months") < 36, "established")
     .otherwise("loyal")
)

# 2. Region mapping
region = (
    F.when(F.col("country") == "IN", "APAC")
     .when(F.col("country") == "AE", "MEA")
     .when(F.col("country") == "UK", "EMEA")
     .when(F.col("country") == "US", "AMER")
     .otherwise("OTHER")
)

# 3. Priority: combined condition first, then the broader one
priority = (
    F.when((F.col("status_code") == "A") & (F.col("monthly_spend") >= 100), "P1")
     .when(F.col("status_code") == "A", "P2")
     .otherwise("P3")
)

result = customers.withColumns({"tenure_group": tenure_group, "region": region, "priority": priority})
result.select("customer_id", "tenure_months", "tenure_group", "country", "region", "status_code", "priority").show()

# 4. SQL version of task 1
customers.selectExpr(
    "customer_id",
    "CASE WHEN tenure_months < 12 THEN 'new' WHEN tenure_months < 36 THEN 'established' ELSE 'loyal' END AS tenure_group",
).show()

# 5. Conditional aggregation
result.groupBy("tenure_group").agg(
    F.count("*").alias("customers"),
    F.sum(F.when(F.col("country") == "IN", 1).otherwise(0)).alias("from_india"),
).show()

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li><code>F.when(cond, val).when(...).otherwise(default)</code> is if/elif/else for columns</li><li>First true condition wins, so order from most specific to most general</li><li>No <code>otherwise</code> means null for unmatched rows</li><li>Null conditions are treated as not true. Add an explicit <code>isNull()</code> branch when needed</li><li>SQL <code>CASE WHEN</code> compiles to the same <code>CaseWhen</code> expression</li><li><code>sum(when(...))</code> and <code>count(when(...))</code> give conditional counts in one pass</li><li>Never use a Python UDF for logic that <code>when</code> can express</li></ul>
</div>

| Task | Code |
|---|---|
| If / else | `F.when(c, a).otherwise(b)` |
| Multiple | `F.when(c1, a).when(c2, b).otherwise(c)` |
| SQL | `CASE WHEN c1 THEN a WHEN c2 THEN b ELSE c END` |
| Simple CASE | `CASE code WHEN 'A' THEN 'Active' ELSE 'Other' END` |
| Conditional count | `F.sum(F.when(c, 1).otherwise(0))` |

## Git commit

```
git add 01_spark_basics/08_when_otherwise.ipynb
git commit -m "add 08 when otherwise notebook"
```